In [ ]:
# Cell 1: Imports

import os
import glob
import joblib
import pandas as pd
import numpy as np

from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
# Cell 2: Dataset path

DATA_PATH = "./data"

csv_files = sorted(
    glob.glob(os.path.join(DATA_PATH, "ddinter_downloads_code_*.csv"))
)

print(f"DDInter CSV files found: {len(csv_files)}")

DDInter CSV files found: 8


In [3]:
# Cell 3: Load all DDInter 2.0 CSV files

ddinter_parts = [
    pd.read_csv(file)
    for file in csv_files
]

ddinter = pd.concat(
    ddinter_parts,
    ignore_index=True
)

print(f"Total records loaded: {len(ddinter)}")

Total records loaded: 222383


In [4]:
# Cell 4: Inspect dataset structure

print("Columns:")
print(ddinter.columns.tolist())

print("\nShape:")
print(ddinter.shape)

print("\nFirst 5 records:")
display(ddinter.head())

Columns:
['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']

Shape:
(222383, 5)

First 5 records:


,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level
0,DDInter1263,Naltrexone,DDInter1,Abacavir,Moderate
1,DDInter1,Abacavir,DDInter1348,Orlistat,Moderate
2,DDInter58,Aluminum hydroxide,DDInter582,Dolutegravir,Major
3,DDInter112,Aprepitant,DDInter582,Dolutegravir,Minor
4,DDInter138,Attapulgite,DDInter582,Dolutegravir,Major


In [5]:
# Cell 5: Check missing values and duplicates

print("Missing values:")
display(ddinter.isnull().sum())

print("\nDuplicate rows:")
print(ddinter.duplicated().sum())

Missing values:


DDInterID_A    0
Drug_A         0
DDInterID_B    0
Drug_B         0
Level          0
dtype: int64


Duplicate rows:
62148


In [6]:
# Cell 6: Check unique drugs

unique_drugs = pd.unique(
    pd.concat([
        ddinter["Drug_A"],
        ddinter["Drug_B"]
    ])
)

print(f"Unique drugs: {len(unique_drugs)}")

print(
    f"Unique DDInter IDs: "
    f"{pd.unique(pd.concat([ddinter['DDInterID_A'], ddinter['DDInterID_B']])).size}"
)

Unique drugs: 1939
Unique DDInter IDs: 1939


In [7]:
# Cell 7: Inspect interaction severity labels

level_counts = ddinter["Level"].value_counts()

display(level_counts)

Level
Moderate    130367
Unknown      47182
Major        33896
Minor        10938
Name: count, dtype: int64

In [8]:
# Cell 8: Create order-independent drug-pair identifier

ddinter["pair_key"] = ddinter.apply(
    lambda row: tuple(
        sorted([
            row["DDInterID_A"],
            row["DDInterID_B"]
        ])
    ),
    axis=1
)

print(f"Unique drug pairs: {ddinter['pair_key'].nunique()}")

Unique drug pairs: 160235


In [9]:
# Cell 9: Check whether a drug pair has conflicting severity labels

pair_level_counts = (
    ddinter.groupby("pair_key")["Level"]
    .nunique()
)

conflicting_pairs = (
    pair_level_counts > 1
).sum()

print(f"Pairs with conflicting labels: {conflicting_pairs}")

Pairs with conflicting labels: 0


In [10]:
# Cell 10: Remove duplicate drug-pair records

ddinter_clean = (
    ddinter
    .drop_duplicates(subset="pair_key")
    .drop(columns="pair_key")
    .reset_index(drop=True)
)

print(f"Original records: {len(ddinter)}")
print(f"Clean records: {len(ddinter_clean)}")

Original records: 222383
Clean records: 160235


In [11]:
# Cell 11: Verify labels after deduplication

print("Interaction levels after cleaning:")
display(
    ddinter_clean["Level"].value_counts()
)

Interaction levels after cleaning:


Level
Moderate    96675
Unknown     29813
Major       26914
Minor        6833
Name: count, dtype: int64

In [12]:
# Cell 12: Save cleaned DDInter 2.0 dataset

CLEAN_PATH = "./data/DDInter2_clean.csv"

ddinter_clean.to_csv(
    CLEAN_PATH,
    index=False
)

print(f"Saved cleaned dataset to: {CLEAN_PATH}")

Saved cleaned dataset to: ./data/DDInter2_clean.csv


## Feature Preparation

In [13]:
# Cell 13: Build unique DDInter drug list

drug_table = pd.concat([
    ddinter_clean[["DDInterID_A", "Drug_A"]].rename(
        columns={"DDInterID_A": "Drug_ID", "Drug_A": "Drug_Name"}
    ),
    ddinter_clean[["DDInterID_B", "Drug_B"]].rename(
        columns={"DDInterID_B": "Drug_ID", "Drug_B": "Drug_Name"}
    )
]).drop_duplicates("Drug_ID").sort_values("Drug_ID").reset_index(drop=True)

print(f"Unique drugs: {len(drug_table)}")
display(drug_table.head())

Unique drugs: 1939


,Drug_ID,Drug_Name
0,DDInter1,Abacavir
1,DDInter10,Acamprosate
2,DDInter100,Anthrax vaccine
3,DDInter1000,Ivosidenib
4,DDInter1001,Ixabepilone


In [14]:
# Cell 14: Encode DDInter interaction severity

level_mapping = {
    "Minor": 0,
    "Moderate": 1,
    "Major": 2,
    "Unknown": 3
}

ddinter_clean["Level_ID"] = (
    ddinter_clean["Level"]
    .map(level_mapping)
    .astype(int)
)

print("Encoded labels:")
display(
    ddinter_clean[["Level", "Level_ID"]]
    .drop_duplicates()
    .sort_values("Level_ID")
)

Encoded labels:


,Level,Level_ID
3,Minor,0
0,Moderate,1
2,Major,2
41600,Unknown,3


In [ ]:
# Cell 15: Stratified train/validation/test split

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    ddinter_clean,
    test_size=0.30,
    stratify=ddinter_clean["Level_ID"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Level_ID"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain class distribution:")
print(train_df["Level"].value_counts(normalize=True).round(4))

print("\nValidation class distribution:")
print(val_df["Level"].value_counts(normalize=True).round(4))

print("\nTest class distribution:")
print(test_df["Level"].value_counts(normalize=True).round(4))

Train: 112164
Validation: 24035
Test: 24036

Train class distribution:
Level
Moderate    0.6033
Unknown     0.1861
Major       0.1680
Minor       0.0426
Name: proportion, dtype: float64

Validation class distribution:
Level
Moderate    0.6033
Unknown     0.1861
Major       0.1680
Minor       0.0426
Name: proportion, dtype: float64

Test class distribution:
Level
Moderate    0.6033
Unknown     0.1861
Major       0.1680
Minor       0.0426
Name: proportion, dtype: float64


In [ ]:
# Cell 16:Build leakage-safe drug interaction features from the training set

drug_train_counts = pd.concat([
    train_df["DDInterID_A"],
    train_df["DDInterID_B"]
]).value_counts()

drug_train_degree = drug_train_counts.to_dict()

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

for df in [train_df, val_df, test_df]:
    df["Drug_A_Degree"] = df["DDInterID_A"].map(drug_train_degree).fillna(0)
    df["Drug_B_Degree"] = df["DDInterID_B"].map(drug_train_degree).fillna(0)
    df["Degree_Difference"] = (
        df["Drug_A_Degree"] - df["Drug_B_Degree"]
    ).abs()

print("Feature columns:")
print(["Drug_A_Degree", "Drug_B_Degree", "Degree_Difference"])

display(
    train_df[
        ["Drug_A", "Drug_B", "Level",
         "Drug_A_Degree", "Drug_B_Degree", "Degree_Difference"]
    ].head()
)

Feature columns:
['Drug_A_Degree', 'Drug_B_Degree', 'Degree_Difference']


,Drug_A,Drug_B,Level,Drug_A_Degree,Drug_B_Degree,Degree_Difference
65107,Cyanocobalamin,Gemcitabine,Unknown,220,239,19
13982,Ceritinib,Lactitol,Moderate,418,142,276
33932,Paroxetine,Rolapitant,Moderate,237,140,97
144635,Betaxolol (ophthalmic),Dyphylline,Major,54,60,6
90186,Docetaxel,Fosaprepitant,Moderate,405,113,292


In [ ]:
# Cell 17: Build training interaction graph and calculate common-neighbor features

from collections import defaultdict

neighbors = defaultdict(set)

for _, row in train_df.iterrows():
    a, b = row["DDInterID_A"], row["DDInterID_B"]
    neighbors[a].add(b)
    neighbors[b].add(a)

def common_neighbors(row):
    a = row["DDInterID_A"]
    b = row["DDInterID_B"]
    return len(neighbors[a] & neighbors[b])

for df in [train_df, val_df, test_df]:
    df["Common_Neighbors"] = df.apply(common_neighbors, axis=1)

print("Common-neighbor feature created.")
print("\nTrain statistics:")
print(train_df["Common_Neighbors"].describe())

display(
    train_df[
        ["Drug_A", "Drug_B", "Level", "Common_Neighbors"]
    ].head()
)


Common-neighbor feature created.

Train statistics:
count    112164.000000
mean         50.480448
std          40.551015
min           0.000000
25%          22.000000
50%          41.000000
75%          67.000000
max         347.000000
Name: Common_Neighbors, dtype: float64


,Drug_A,Drug_B,Level,Common_Neighbors
65107,Cyanocobalamin,Gemcitabine,Unknown,84
13982,Ceritinib,Lactitol,Moderate,78
33932,Paroxetine,Rolapitant,Moderate,23
144635,Betaxolol (ophthalmic),Dyphylline,Major,8
90186,Docetaxel,Fosaprepitant,Moderate,54


In [ ]:
# Cell 18: Build leakage-safe drug severity profiles

severity_levels = ["Minor", "Moderate", "Major", "Unknown"]

drug_label_data = pd.concat([
    train_df[["DDInterID_A", "Level"]].rename(
        columns={"DDInterID_A": "Drug_ID"}
    ),
    train_df[["DDInterID_B", "Level"]].rename(
        columns={"DDInterID_B": "Drug_ID"}
    )
])

drug_counts = pd.crosstab(
    drug_label_data["Drug_ID"],
    drug_label_data["Level"]
).reindex(columns=severity_levels, fill_value=0)

drug_profile = drug_counts.div(
    drug_counts.sum(axis=1), axis=0
)

drug_profile.columns = [
    f"Drug_Profile_{level}" for level in severity_levels
]


def add_training_profiles(df):
    result = df.copy()

    for side, id_col in [("A", "DDInterID_A"), ("B", "DDInterID_B")]:
        for level in severity_levels:
            values = []

            for drug_id, label in zip(df[id_col], df["Level"]):
                counts = drug_counts.loc[drug_id].copy()
                counts[label] -= 1

                total = counts.sum()
                values.append(
                    counts[level] / total if total > 0 else 0
                )

            result[f"{side}_Profile_{level}"] = values

    return result


# Training: leave-one-out profiles
train_df = add_training_profiles(train_df)

# Validation/Test: training-set profiles only
for df in [val_df, test_df]:
    for side, id_col in [("A", "DDInterID_A"), ("B", "DDInterID_B")]:
        for level in severity_levels:
            profile_col = f"{side}_Profile_{level}"

            df[profile_col] = df[id_col].map(
                drug_profile[f"Drug_Profile_{level}"]
            ).fillna(0)

print("Leakage-safe severity profiles created.")

Leakage-safe severity profiles created.


In [57]:
# Cell 19: Prepare final model features and targets

feature_cols_v2 = [
    "Drug_A_Degree",
    "Drug_B_Degree",
    "Degree_Difference",
    "Common_Neighbors",
    "A_Profile_Minor",
    "A_Profile_Moderate",
    "A_Profile_Major",
    "A_Profile_Unknown",
    "B_Profile_Minor",
    "B_Profile_Moderate",
    "B_Profile_Major",
    "B_Profile_Unknown"
]

X_train_v2 = train_df[feature_cols_v2].values
X_val_v2 = val_df[feature_cols_v2].values
X_test_v2 = test_df[feature_cols_v2].values

y_train = train_df["Level_ID"].values
y_val = val_df["Level_ID"].values
y_test = test_df["Level_ID"].values

print("X_train:", X_train_v2.shape)
print("X_val:", X_val_v2.shape)
print("X_test:", X_test_v2.shape)
print("Targets:", y_train.shape, y_val.shape, y_test.shape)

X_train: (112164, 12)
X_val: (24035, 12)
X_test: (24036, 12)
Targets: (112164,) (24035,) (24036,)


In [ ]:
# Cell 20: Scale the improved 12-feature dataset

scaler_v2 = StandardScaler()

X_train_v2_scaled = scaler_v2.fit_transform(X_train_v2)
X_val_v2_scaled = scaler_v2.transform(X_val_v2)
X_test_v2_scaled = scaler_v2.transform(X_test_v2)

print("V2 scaling complete.")
print("Training means:", X_train_v2_scaled.mean(axis=0).round(4))
print("Training std:", X_train_v2_scaled.std(axis=0).round(4))

V2 scaling complete.
Training means: [ 0.  0.  0. -0.  0. -0.  0.  0. -0. -0. -0. -0.]
Training std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
# Cell 21: Train improved V2 Random Forest

model_v2 = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
    max_features="sqrt"
)

model_v2.fit(X_train_v2_scaled, y_train)

print("V2 model training complete.")

V2 model training complete.


In [ ]:
# Cell 22: Evaluate V2 on the validation set

y_val_pred_v2 = model_v2.predict(X_val_v2_scaled)

print("V2 Validation Metrics")
print("----------------------")
print(f"Accuracy:           {accuracy_score(y_val, y_val_pred_v2):.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_score(y_val, y_val_pred_v2):.4f}")
print(f"Macro Precision:    {precision_score(y_val, y_val_pred_v2, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:       {recall_score(y_val, y_val_pred_v2, average='macro', zero_division=0):.4f}")
print(f"Macro F1:           {f1_score(y_val, y_val_pred_v2, average='macro', zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_v2,
    target_names=["Minor", "Moderate", "Major", "Unknown"],
    zero_division=0
))

V2 Validation Metrics
----------------------
Accuracy:           0.8771
Balanced Accuracy:  0.7664
Macro Precision:    0.8879
Macro Recall:       0.7664
Macro F1:           0.8096

Classification Report:
              precision    recall  f1-score   support

       Minor       0.93      0.51      0.66      1025
    Moderate       0.88      0.94      0.91     14501
       Major       0.89      0.72      0.80      4037
     Unknown       0.86      0.89      0.87      4472

    accuracy                           0.88     24035
   macro avg       0.89      0.77      0.81     24035
weighted avg       0.88      0.88      0.87     24035



In [ ]:
# Cell 23: Final evaluation of V2 on the untouched test set

y_test_pred_v2 = model_v2.predict(X_test_v2_scaled)

v2_accuracy = accuracy_score(y_test, y_test_pred_v2)
v2_balanced_acc = balanced_accuracy_score(y_test, y_test_pred_v2)
v2_macro_precision = precision_score(
    y_test, y_test_pred_v2, average="macro", zero_division=0
)
v2_macro_recall = recall_score(
    y_test, y_test_pred_v2, average="macro", zero_division=0
)
v2_macro_f1 = f1_score(
    y_test, y_test_pred_v2, average="macro", zero_division=0
)
v2_weighted_f1 = f1_score(
    y_test, y_test_pred_v2, average="weighted", zero_division=0
)

print("FINAL V2 TEST METRICS")
print("---------------------")
print(f"Accuracy:           {v2_accuracy:.4f}")
print(f"Balanced Accuracy:  {v2_balanced_acc:.4f}")
print(f"Macro Precision:    {v2_macro_precision:.4f}")
print(f"Macro Recall:       {v2_macro_recall:.4f}")
print(f"Macro F1:           {v2_macro_f1:.4f}")
print(f"Weighted F1:        {v2_weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred_v2,
    target_names=["Minor", "Moderate", "Major", "Unknown"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_v2))

FINAL V2 TEST METRICS
---------------------
Accuracy:           0.8760
Balanced Accuracy:  0.7636
Macro Precision:    0.8848
Macro Recall:       0.7636
Macro F1:           0.8058
Weighted F1:        0.8719

Classification Report:
              precision    recall  f1-score   support

       Minor       0.92      0.49      0.64      1025
    Moderate       0.88      0.94      0.91     14502
       Major       0.87      0.73      0.79      4037
     Unknown       0.86      0.89      0.88      4472

    accuracy                           0.88     24036
   macro avg       0.88      0.76      0.81     24036
weighted avg       0.88      0.88      0.87     24036


Confusion Matrix:
[[  505   410    15    95]
 [   37 13608   373   484]
 [    3  1032  2943    59]
 [    1   430    41  4000]]


In [ ]:
# Cell 24: Save the final DDInter V2 model and preprocessing objects

import joblib
import os

os.makedirs("./models", exist_ok=True)

joblib.dump(model_v2, "./models/ddinter_v2_random_forest.pkl")
joblib.dump(scaler_v2, "./models/ddinter_v2_scaler.pkl")
joblib.dump(feature_cols_v2, "./models/ddinter_v2_features.pkl")

print("Final model saved successfully.")
print("Saved files:")
print("- ./models/ddinter_v2_random_forest.pkl")
print("- ./models/ddinter_v2_scaler.pkl")
print("- ./models/ddinter_v2_features.pkl")

Final model saved successfully.
Saved files:
- ./models/ddinter_v2_random_forest.pkl
- ./models/ddinter_v2_scaler.pkl
- ./models/ddinter_v2_features.pkl
